In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import rasterio
import os
from rasterio import features

In [ ]:
with rasterio.open('data/bol_pop_2025.tif') as src:
    prf = src.profile
    trf = src.transform
    arr = src.read(1)
    bds = src.bounds

adm = gpd.read_file('data/bol_gadm.json')

val = [(geom, i+1) for i,geom in zip(adm.index.values, adm.geometry)]
mst = features.rasterize(
    val,
    out_shape=(prf['height'], prf['width']),
    transform=trf,
    fill=-1,
    dtype=np.int16
)

df = pd.DataFrame({'adm': mst.flatten(), 'pop': arr.flatten()})
sel = np.logical_and(df['adm'] > 0, df['pop'] > 0)
df = df[sel].groupby('adm').sum().reset_index(drop=False)

df.to_csv('data/bol_pop_2025.csv', index=False)
adm_ = adm[['geometry']].copy()
adm_['adm'] = adm.index.values + 1
mrg = pd.merge(df, adm_, on='adm', how='outer')
mrg = gpd.GeoDataFrame(mrg, geometry='geometry', crs=adm.crs)
mrg.to_file('data/bol_pop_2025.gpkg', index=False)

p = prf.copy()
p.update(dtype='int16', count=1, nodata=-1)
with rasterio.open('data/bol_gadm.tif', 'w', **p) as dst:
    dst.write(mst, 1)

p = prf.copy()
arr_ = (arr.copy() > 0).astype(np.uint8)
p.update(dtype='uint8', count=1, nodata=0)
with rasterio.open('data/bol_mask.tif', 'w', **p) as dst:
    dst.write(arr_, 1)

,pop
adm,
1,228212.000000
2,119479.601562
3,44455.816406
4,26225.888672
5,35511.324219


In [170]:
import importlib
import dasymetric 
from dasymetric import DasymetricConfig, DasymetricRedistributor

In [ ]:
dasymetric._rasterize_zones()

In [217]:
importlib.reload(dasymetric)
from dasymetric import DasymetricConfig, DasymetricRedistributor

config = DasymetricConfig(
    weight_raster_path="data/bol_viirs_2023.tif",
    pop_path="data/bol_pop_2025.csv",
    #geom_path="out/test_pop_mastergrid.tif",
    geom_path="data/bol_pop_2025.gpkg",
    output_raster_path="out/test_pop.tif",
    pop_field="pop",
    mask_path="data/bol_mask.tif",
    id_field="adm",
    nodata=-99999,
    #id_field="idx",     # optional; auto-generated if omitted
    block_size=1024,
    n_workers=4,
    max_windows=128
)

job = DasymetricRedistributor(config)
#job.run()

2026-07-17 22:30:57,888 [INFO] Creating mastergrid from vector layer...
2026-07-17 22:30:57,900 [INFO] Rasterizing mastergrid in parallel (240 windows, 4 workers)...


ValueError: too many values to unpack (expected 2)

In [219]:
[a for a in dasymetric.as_completed(job.futures)][10].result()

(Window(col_off=0, row_off=4096, width=1024, height=1024),
 array([[-1, -1, -1, ..., -1, -1, -1],
        [-1, -1, -1, ..., -1, -1, -1],
        [-1, -1, -1, ..., -1, -1, -1],
        ...,
        [-1, -1, -1, ..., -1, -1, -1],
        [-1, -1, -1, ..., -1, -1, -1],
        [-1, -1, -1, ..., -1, -1, -1]], shape=(1024, 1024), dtype=int32))

In [157]:
b

array([ 1.80839206,  2.9175684 ,  3.70204893,  5.15732796,  3.62899888,
        3.22276804,  1.14019839,  3.3394179 ,  5.81987744,  3.29453191,
        1.5665935 ,  0.86527535,  4.11984266,  0.67892671,  2.16513796,
        2.59812627,  2.46689426,  2.94532122,  0.68700506,  3.89712938,
        3.47948371,  3.82132467,  1.86966981,  0.87993004,  1.40816409,
        0.73524374,  0.76505465,  0.90828812,  0.59368238,  2.79853224,
        0.59310073,  0.6656355 ,  1.89969021,  1.75218689,  2.83021325,
        1.09218762, 11.55465906,  3.67893651,  4.92430819,  3.92782803,
        3.94987711,  3.06284782,  0.44009162,  4.28360324,  3.76494694,
        6.33794764,  1.03982156,  1.38423309, 12.72870157,  1.00671344,
       11.43809988,  1.78329496,  2.05311037,  5.90581509,  0.92982062,
        1.57057907,  0.37074977,  1.16456202,  1.69240008,  1.11927125,
        1.32377987,  0.82244748,  0.72941922,  0.93046644,  1.38293729,
        4.9453445 ,  3.04493568,  1.93381525,  1.53522685,  0.92

In [149]:
job.config.id_field

'adm'